# ETA Model Training (GPU + LGBM)

Train NN on full 37M rows using T4 GPU, then train LightGBM on CPU.
Works on both Google Colab and Kaggle.

In [ ]:
# 1. Install dependencies (torch is pre-installed on Colab/Kaggle)
!pip install -q huggingface_hub pyarrow tqdm mlflow lightgbm

In [ ]:
# 2. Clone repo
!git clone https://github.com/sarthakbiswas97/eta-engine.git
%cd eta-engine

In [ ]:
# 3. Download data directly from NYC TLC + compute zone-pair stats
!pip install -q -r requirements.txt
!python data/download_data.py
!python -m features.zone_pair_stats

In [ ]:
# 4. Verify GPU and data
import torch
import os

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    vram = getattr(props, "total_memory", getattr(props, "total_mem", 0)) / 1e9
    print(f"GPU: {gpu}")
    print(f"VRAM: {vram:.1f} GB")

print()
for f in ["data/train.parquet", "data/dev.parquet", "data/zone_pair_stats/zone_pair_stats.pkl"]:
    size = os.path.getsize(f) / 1e6 if os.path.exists(f) else 0
    status = f"{size:.1f} MB" if size > 0 else "MISSING"
    print(f"  {f}: {status}")

In [ ]:
# 5a. Train NN (GPU)
!python train.py --epochs 10 --batch-size 8192 --lr 5e-4 --patience 3 --num-workers 2 --dev-sample 50000 --save-every 2 --loss huber --run-name v4b-huber

In [ ]:
# 5b. Train LightGBM (CPU, ~5-10 min on 10M rows)
!python scripts/train_lgbm.py --sample 10000000 --dev-sample 100000 --run-name lgbm-v1

In [ ]:
# 5c. Find optimal ensemble weight
!python scripts/find_ensemble_weight.py --dev-sample 0

In [ ]:
# 6. Check NN results
checkpoint = torch.load("model.pt", map_location="cpu", weights_only=False)
print(f"NN Best dev MAE: {checkpoint['dev_mae']:.1f} s")
print(f"Best epoch: {checkpoint['epoch']}")
print(f"Model config: {checkpoint['model_config']}")
print(f"Log target: {checkpoint.get('log_target', False)}")

# Check LGBM
import lightgbm as lgb
lgbm = lgb.Booster(model_file="lgbm_model.txt")
print(f"\nLGBM trees: {lgbm.num_trees()}")
print(f"LGBM model size: {os.path.getsize('lgbm_model.txt') / 1e6:.1f} MB")

In [ ]:
# 7. Upload models to HF Hub
from huggingface_hub import HfApi

MODEL_REPO = "sarthakbiswas/eta-engine"
VERSION = "v4b"  # <-- change this each run

# Get HF token
token = os.environ.get("HF_TOKEN")
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        pass
if not token:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass

if token:
    api = HfApi(token=token)
    api.create_repo(repo_id=MODEL_REPO, exist_ok=True)

    # NN model (versioned + current)
    api.upload_file(path_or_fileobj="model.pt", path_in_repo=f"model_{VERSION}.pt", repo_id=MODEL_REPO)
    api.upload_file(path_or_fileobj="model.pt", path_in_repo="model.pt", repo_id=MODEL_REPO)
    print(f"Uploaded NN model_{VERSION}.pt + model.pt")

    # LGBM model
    api.upload_file(path_or_fileobj="lgbm_model.txt", path_in_repo="lgbm_model.txt", repo_id=MODEL_REPO)
    print("Uploaded lgbm_model.txt")

    print(f"https://huggingface.co/{MODEL_REPO}")
else:
    print("No HF_TOKEN found.")

In [ ]:
# 8. Archive MLflow runs to HF Hub
import tarfile

mlruns_tar = "mlruns.tar.gz"
with tarfile.open(mlruns_tar, "w:gz") as tar:
    tar.add("mlruns", arcname="mlruns")
print(f"Archived mlruns to {mlruns_tar}")

if token:
    api.upload_file(path_or_fileobj=mlruns_tar, path_in_repo="mlruns.tar.gz", repo_id=MODEL_REPO)
    print("MLflow runs uploaded")
else:
    print("No HF_TOKEN.")